# 🔀 使用 Microsoft Foundry 的条件代理工作流（Python）

## 📋 基于高级决策的工作流教程

本笔记本示范了使用 Microsoft Foundry 和 Microsoft Agent Framework 的<strong>条件工作流模式</strong>。您将学习如何构建智能的、基于决策的工作流，这些工作流根据内容分析、业务规则和 AI 驱动的决策动态路由处理流程。

## 🎯 学习目标

### 🧠 <strong>智能决策</strong>
- <strong>条件逻辑</strong>：基于 AI 分析和业务规则实现动态分支
- <strong>内容感知路由</strong>：根据内容分析和分类路由工作流路径
- <strong>自适应处理</strong>：根据实时条件和数据调整工作流行为
- **Azure AI 集成**：充分利用 Microsoft Foundry 的高级决策功能

### 🔀 <strong>高级工作流模式</strong>
- <strong>决策树</strong>：构建多分支的复杂决策结构
- <strong>基于规则的处理</strong>：实现业务逻辑和合规要求
- <strong>动态工作流修改</strong>：基于运行时条件调整工作流
- <strong>上下文感知操作</strong>：基于累计的工作流上下文做出决策

### 🏢 <strong>企业条件应用</strong>
- <strong>文档分类</strong>：将文档路由到适当的处理工作流
- <strong>客户服务分诊</strong>：自动将咨询路由至专项处理工作流
- <strong>合规处理</strong>：根据内容类型和法规应用不同验证规则
- <strong>质量保证</strong>：根据质量指标将内容路由至不同审核流程

## ⚙️ 前置条件与设置

### 📦 <strong>安装与依赖</strong>

此工作流需要针对 Microsoft Foundry 集成执行特定安装步骤：

```bash

pip install agent-framework-azure-ai -U 
```

### 🔑 **Microsoft Foundry 配置**

**所需 Azure 资源：**
- 部署了相应模型的 Microsoft Foundry 工作区
- 具备必要权限的 Azure 订阅
- 配置好的 Azure CLI 认证


**身份验证设置：**
```bash
# Azure CLI 身份验证
az login
az account set --subscription "your-subscription-id"
azd auth login
```

### 🏗️ <strong>条件工作流架构</strong>

```mermaid
graph TD
    A[输入文档/请求] --> B[初步分析代理]
    B --> C{决策点}
    C -->|条件1| D[工作流程路径A]
    C -->|条件2| E[工作流程路径B]
    C -->|条件3| F[工作流程路径C]
    D --> G[专门处理A]
    E --> H[专门处理B]
    F --> I[专门处理C]
    G --> J[结果整合]
    H --> J
    I --> J
    J --> K[最终输出]
```

**关键组件：**
- <strong>分析代理</strong>：评估内容并做出路由决策的 AI 代理
- <strong>决策点</strong>：确定工作流路径的条件逻辑
- <strong>专项处理器</strong>：针对特定内容类型或场景优化的不同代理
- <strong>集成层</strong>：汇总不同工作流路径结果

## 🎨 <strong>条件工作流设计模式</strong>

### 📋 <strong>文档处理分诊</strong>
```
Document Input → Content Analysis → Classification → Specialized Processing Workflow
```

### 🎯 <strong>客户服务路由</strong>
```
Customer Inquiry → Intent Analysis → Urgency Assessment → Route to Specialist Team
```

### 🔍 <strong>质量保证工作流</strong>
```
Content Input → Quality Metrics → Risk Assessment → Appropriate Review Process
```

### 📊 <strong>业务智能管道</strong>
```
Data Input → Source Analysis → Processing Rules → Specialized Analytics Workflow
```

## 🏢 <strong>企业收益</strong>

### 🎯 <strong>智能自动化</strong>
- <strong>智能路由</strong>：自动将工作导向最合适的处理路径
- <strong>自适应行为</strong>：基于模式和结果学习与调整的工作流
- <strong>业务规则集成</strong>：融合复杂业务逻辑和合规要求
- <strong>上下文感知处理</strong>：基于完整工作流上下文和历史做决策

### 📈 <strong>运营效率</strong>
- <strong>减少人工干预</strong>：自动决策减少人工路由需求
- <strong>专项处理</strong>：为特定场景优化的各条工作流路径
- <strong>资源优化</strong>：根据内容类型高效分配处理资源
- <strong>更快解决时间</strong>：直接路由到相应专家和流程

### 🛡️ <strong>治理与控制</strong>
- <strong>审计日志</strong>：完整记录决策点和路由依据
- <strong>合规执行</strong>：自动应用法规和政策要求
- <strong>风险管理</strong>：通过加强安全与审核流程处理高风险内容
- <strong>质量保证</strong>：基于内容特征确保适当的审核级别

### 📊 <strong>分析与优化</strong>
- <strong>决策分析</strong>：跟踪路由决策和工作流路径的有效性
- <strong>性能指标</strong>：衡量各工作流分支的效率
- <strong>持续改进</strong>：发现条件逻辑的优化机会
- <strong>业务智能</strong>：洞察内容模式和处理需求

让我们构建智能、基于决策的 AI 工作流吧！🚀


In [ ]:
! pip install agent-framework-azure-ai -U 

requirements.txt 和 constraints.txt - 在 ./Installation 目录中

请将 .env.examples 复制为 .env


<strong>注意</strong> 请选择 gpt-4.1-mini


In [ ]:
import os

from dataclasses import dataclass
from typing_extensions import Literal
from pydantic import BaseModel

In [ ]:
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

from agent_framework import HostedWebSearchTool
from agent_framework.azure import AzureAIAgentClient
from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    HostedCodeInterpreterTool,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    executor,
    WorkflowViz
)


from azure.ai.agents.models import BingGroundingTool,CodeInterpreterTool

In [ ]:
load_dotenv()

In [ ]:
EvangelistInstructions = """
You are a technology evangelist create a first draft for a technical tutorials.
1. Each knowledge point in the outline must include a link. Follow the link to access the content related to the knowledge point in the outline. Expand on that content.
2. Each knowledge point must be explained in detail.
3. Rewrite the content according to the entry requirements, including the title, outline, and corresponding content. It is not necessary to follow the outline in full order.
4. The content must be more than 200 words.
4. Output draft as Markdown format. set 'draft_content' to the draft content.
5. return result as JSON with fields 'draft_content' (string).
"""

ContentReviewerInstructions = """
You are a content reviewer for a publishing company. You need to check whether the tutorial's draft content meets the following requirements:

1. The draft content less than 200 words, set 'review_result' to 'No' and 'reason' to 'Content is too short'. If the draft content is more than 200 words, set 'review_result' to 'Yes' and 'reason' to 'The content is good'.
2. set 'draft_content' to the original draft content.
3. return result as JSON with fields 'review_result' (one of Yes, No) and 'reason' (string) and 'draft_content' (string).

"""

PublisherInstructions = """
You are the content publisher ,run code to save the tutorial's draft content as a Markdown file. Saved file's name is marked with current date and time, such as yearmonthdayhourminsec. Note that if it is 1-9, you need to add 0, such as  20240101123045.md. 
"""

In [ ]:
OUTLINE_Content ="""
# Introduce AI Agent


## What's AI Agent

https://github.com/microsoft/ai-agents-for-beginners/tree/main/01-intro-to-ai-agents


***Note*** Don's create any sample code 


## Introduce Microsoft Foundry Agent Service 

https://learn.microsoft.com/en-us/azure/ai-foundry/agents/overview


***Note*** Don's create any sample code 


## Microsoft Agent Framework 

https://github.com/microsoft/agent-framework/tree/main/docs/docs-templates


***Note*** Don's create any sample code 
"""

In [ ]:
conn_id = os.environ["BING_CONNECTION_ID"]  # Ensure the BING_CONNECTION_NAME environment variable is set

# Initialize the Bing Grounding tool
bing = BingGroundingTool(connection_id=conn_id)

code_interpreter = CodeInterpreterTool()

In [ ]:
class EvangelistAgent(BaseModel):
    draft_content: str

class ReviewAgent(BaseModel):
    review_result: Literal["Yes", "No"]
    reason: str
    draft_content: str

class PublisherAgent(BaseModel):
    file_path: str

@dataclass
class ReviewResult:
    review_result: str
    reason: str
    draft_content: str

@executor(id="to_reviewer_result")
async def to_reviewer_result(response: AgentExecutorResponse, ctx: WorkflowContext[ReviewResult]) -> None:

    print(f"Raw response from reviewer agent: {response.agent_run_response.text}")

    parsed = ReviewAgent.model_validate_json(response.agent_run_response.text)
    await ctx.send_message(
        ReviewResult(
            review_result=parsed.review_result,
            reason=parsed.reason,
            draft_content=parsed.draft_content,
        )
    )


def select_targets(review: ReviewResult, target_ids: list[str]) -> list[str]:
        # Order: [handle_review, submit_to_email_assistant, summarize_email, handle_uncertain]
        handle_review_id, save_draft_id = target_ids
        if review.review_result == "Yes":
            return [save_draft_id]
        else:
            return [handle_review_id]
        


@executor(id="handle_review")
async def handle_review(review: ReviewResult, ctx: WorkflowContext[str]) -> None:
    if review.review_result == "No":
        await ctx.yield_output(f"Review failed: {review.reason}, please revise the draft.")
    else:
        await ctx.send_message(
            AgentExecutorRequest(messages=[ChatMessage(Role.USER, text=review.draft_content)], should_respond=True)
        )


@executor(id="save_draft")
async def save_draft(review: ReviewResult, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    # Only called for long NotSpam emails by selection_func
    await ctx.send_message(
        AgentExecutorRequest(messages=[ChatMessage(Role.USER, text=review.draft_content)], should_respond=True)
    )


In [ ]:
from IPython.display import SVG, display, HTML

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):  
        try:
                evangelist_agent = AgentExecutor(chat_client.create_agent(
                    instructions= (EvangelistInstructions),
                    tools=[HostedWebSearchTool()],
                    # response_format=EvangelistAgent
                ),  id="evangelist_agent")
                reviewer_agent = AgentExecutor(chat_client.create_agent(
                    instructions=(ContentReviewerInstructions),
                    # response_format=ReviewAgent
                ), id="reviewer_agent")
                publisher_agent = AgentExecutor(chat_client.create_agent(
                    instructions=PublisherInstructions,
                    tools=HostedCodeInterpreterTool(),
                    response_format=PublisherAgent
                ), id="publisher_agent")

                workflow = (
                    WorkflowBuilder()
                        .set_start_executor(evangelist_agent)
                        .add_edge(evangelist_agent, reviewer_agent)
                        .add_edge(reviewer_agent, to_reviewer_result)
                        .add_multi_selection_edge_group(
                            to_reviewer_result,
                            [handle_review, save_draft],
                            selection_func=select_targets,
                        )
                        .add_edge(save_draft, publisher_agent)
                        .build()
                )

                # workflow = SequentialBuilder().participants([evangelist_chat_agent, reviewer_chat_agent, publisher_chat_agent]).build()
                print("Generating workflow visualization...")
                viz = WorkflowViz(workflow)
                # Print out the mermaid string.
                print("Mermaid string: \n=======")
                print(viz.to_mermaid())
                print("=======")
                # Print out the DiGraph string.
                print("DiGraph string: \n=======")
                print(viz.to_digraph())
                print("=======")
                svg_file = viz.export(format="svg")
                print(f"SVG file saved to: {svg_file}")

                if svg_file and os.path.exists(svg_file):
                    try:
                        # Preferred: direct SVG rendering
                        display(SVG(filename=svg_file))
                    except Exception as e:
                        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
                        try:
                            with open(svg_file, "r", encoding="utf-8") as f:
                                svg_text = f.read()
                            display(HTML(svg_text))
                        except Exception as inner:
                            print(f"❌ Fallback HTML render also failed: {inner}")
                else:
                    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

                
                task = """
                    You are a evangelist , need to write a  draft based on the following outline and the content provided in the link corresponding to the outline. After draft create , the reviewer check it , if it meets the requirements, it will be submitted to the publisher and save it as a Markdown file, otherwise need to rewrite draft until it meets the requirements.
                        The provided outline content and related links is as follows:

                    """ + OUTLINE_Content

                
                async for event in workflow.run_stream(task):
                    if isinstance(event, DatabaseEvent):
                        print(f"{event}")
                    if isinstance(event, WorkflowEvent):
                        print(f"Workflow output: {event.data}")



        finally:
            print("done")


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
